# Notebook 16: Reflection & Self-Correction for Agents

**Frontier ML Interview Prep** | Agentic Systems Track

---

When an agent fails, the naive fix is to retry. But retrying without learning from the failure leads to the same mistake. Reflexion and self-correction techniques let agents learn from their errors *within a single episode* -- no weight updates, no fine-tuning. This is verbal reinforcement learning: the agent reflects on what went wrong, stores that reflection in memory, and uses it to do better on the next attempt.

This notebook implements the full Reflexion framework, applies it to code debugging, explores constitutional self-critique, and rigorously examines when self-correction fails.

## 1. Self-Quiz (Active Recall)

Before reading, try to answer these from memory:

1. **What is Reflexion?** How does it differ from simply retrying a failed task?
2. **What is verbal reinforcement learning?** How is the "reward signal" represented?
3. **What are the three components** of the Reflexion framework?
4. **When does self-correction fail?** Name at least two failure modes.
5. **How is Reflexion different from RLHF?** What changes between attempts?

In [ ]:
# YOUR ANSWERS (write before reading the notebook):
#
# 1. What is Reflexion?
#    ...
#
# 2. Verbal reinforcement learning:
#    ...
#
# 3. Three components:
#    ...
#
# 4. When self-correction fails:
#    ...
#
# 5. Reflexion vs RLHF:
#    ...

## 2. Setup

In [ ]:
!pip install openai matplotlib -q

In [ ]:
import os
import json
import time
import random
import traceback
import textwrap
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple, Callable
import matplotlib.pyplot as plt
import numpy as np

# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "sk-..."

try:
    from openai import OpenAI
    client = OpenAI()
    USE_LLM = True
    print("OpenAI client initialized. Will use real LLM calls.")
except Exception:
    USE_LLM = False
    print("No OpenAI API key found. Using simulated responses for demonstration.")

print("Setup complete.")

## 3. The Retry Problem

The simplest "error correction" strategy is retry: if the agent fails, run it again. Let's see why this doesn't work.

In [ ]:
# Demonstrate the retry problem with a coding task

@dataclass
class CodingTask:
    """A coding problem with test cases."""
    task_id: str
    description: str
    function_name: str
    test_cases: List[Dict[str, Any]]  # [{"input": ..., "expected": ...}]

    def run_tests(self, code: str) -> Dict[str, Any]:
        """Execute the code against test cases. Returns results."""
        results = {"passed": 0, "failed": 0, "errors": [], "details": []}

        try:
            # Execute the code to define the function
            exec_globals = {}
            exec(code, exec_globals)
            func = exec_globals.get(self.function_name)
            if func is None:
                results["errors"].append(f"Function '{self.function_name}' not defined")
                return results
        except Exception as e:
            results["errors"].append(f"Compilation error: {str(e)}")
            return results

        for i, tc in enumerate(self.test_cases):
            try:
                if isinstance(tc["input"], tuple):
                    actual = func(*tc["input"])
                else:
                    actual = func(tc["input"])
                if actual == tc["expected"]:
                    results["passed"] += 1
                    results["details"].append({"test": i, "status": "PASS"})
                else:
                    results["failed"] += 1
                    results["details"].append({
                        "test": i, "status": "FAIL",
                        "input": tc["input"], "expected": tc["expected"], "actual": actual
                    })
                    results["errors"].append(
                        f"Test {i}: input={tc['input']}, expected={tc['expected']}, got={actual}"
                    )
            except Exception as e:
                results["failed"] += 1
                results["errors"].append(f"Test {i}: Runtime error: {str(e)}")
                results["details"].append({"test": i, "status": "ERROR", "error": str(e)})

        return results


# Define sample tasks
CODING_TASKS = [
    CodingTask(
        task_id="P1",
        description="Write a function `two_sum(nums, target)` that returns indices of the two numbers that add up to target.",
        function_name="two_sum",
        test_cases=[
            {"input": ([2, 7, 11, 15], 9), "expected": [0, 1]},
            {"input": ([3, 2, 4], 6), "expected": [1, 2]},
            {"input": ([3, 3], 6), "expected": [0, 1]},
        ]
    ),
    CodingTask(
        task_id="P2",
        description="Write a function `is_palindrome(s)` that checks if a string is a palindrome, ignoring non-alphanumeric characters and case.",
        function_name="is_palindrome",
        test_cases=[
            {"input": "A man, a plan, a canal: Panama", "expected": True},
            {"input": "race a car", "expected": False},
            {"input": " ", "expected": True},
            {"input": "ab_a", "expected": True},
            # Digit case: isalpha()-based filtering keeps only "p" (True), isalnum() keeps "0p" (False) -- so the planted bug actually fails a test
            {"input": "0P", "expected": False},
        ]
    ),
    CodingTask(
        task_id="P3",
        description="Write a function `max_profit(prices)` that returns maximum profit from buying and selling one stock. Return 0 if no profit possible.",
        function_name="max_profit",
        test_cases=[
            {"input": [7, 1, 5, 3, 6, 4], "expected": 5},
            {"input": [7, 6, 4, 3, 1], "expected": 0},
            {"input": [2, 4, 1], "expected": 2},
        ]
    ),
    CodingTask(
        task_id="P4",
        description="Write a function `valid_parentheses(s)` that checks if parentheses string '()[]{}' is valid.",
        function_name="valid_parentheses",
        test_cases=[
            {"input": "()", "expected": True},
            {"input": "()[]{}", "expected": True},
            {"input": "(]", "expected": False},
            {"input": "([)]", "expected": False},
            {"input": "{[]}", "expected": True},
        ]
    ),
    CodingTask(
        task_id="P5",
        description="Write a function `fizzbuzz(n)` that returns a list of strings from 1 to n. For multiples of 3: 'Fizz', multiples of 5: 'Buzz', both: 'FizzBuzz', otherwise the number as string.",
        function_name="fizzbuzz",
        test_cases=[
            {"input": 3, "expected": ["1", "2", "Fizz"]},
            {"input": 5, "expected": ["1", "2", "Fizz", "4", "Buzz"]},
            {"input": 15, "expected": ["1", "2", "Fizz", "4", "Buzz", "Fizz", "7", "8", "Fizz", "Buzz", "11", "Fizz", "13", "14", "FizzBuzz"]},
        ]
    ),
]

print(f"Defined {len(CODING_TASKS)} coding tasks:")
for t in CODING_TASKS:
    print(f"  {t.task_id}: {t.function_name} ({len(t.test_cases)} test cases)")

In [ ]:
# Demonstrate the RETRY problem: same agent makes the same mistake

# Simulated "buggy" solutions that a naive agent might produce
BUGGY_SOLUTIONS = {
    "P1": '''def two_sum(nums, target):
    # Bug: returns values instead of indices
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            if nums[i] + nums[j] == target:
                return [nums[i], nums[j]]  # BUG: should be [i, j]
    return []
''',
    "P2": '''def is_palindrome(s):
    # Bug: doesn't handle underscores correctly
    cleaned = ''.join(c.lower() for c in s if c.isalpha())
    return cleaned == cleaned[::-1]
    # BUG: isalpha() excludes digits and doesn't include _ handling
    # Should be: c.isalnum()
''',
    "P3": '''def max_profit(prices):
    # Bug: O(n^2) but also wrong - considers selling before buying
    max_p = 0
    for i in range(len(prices)):
        for j in range(len(prices)):  # BUG: j should start from i+1
            max_p = max(max_p, prices[j] - prices[i])
    return max_p
''',
}

print("Demonstrating the RETRY problem:\n")
print("We have a buggy solution for P1 (two_sum). Let's 'retry' it 3 times.")
print("Since the agent doesn't learn from failures, it makes the SAME mistake.\n")

task = CODING_TASKS[0]  # two_sum
buggy_code = BUGGY_SOLUTIONS["P1"]

for attempt in range(1, 4):
    print(f"--- Attempt {attempt} ---")
    results = task.run_tests(buggy_code)
    print(f"  Passed: {results['passed']}/{results['passed'] + results['failed']}")
    for err in results["errors"]:
        print(f"  Error: {err}")
    print()

print("Result: All 3 attempts produce the EXACT SAME error.")
print("The agent returns values [2, 7] instead of indices [0, 1].")
print("\nKey insight: Without LEARNING from the failure, retry is useless.")
print("The agent needs to REFLECT on WHY it failed and CHANGE its approach.")

## 4. Reflexion Framework

**Paper**: [Reflexion: Language Agents with Verbal Reinforcement Learning](https://arxiv.org/abs/2303.11366) (Shinn et al., 2023)

Reflexion has three components:

1. **Actor**: Generates actions (code, tool calls, etc.) given a task and past reflections
2. **Evaluator**: Scores the outcome (run tests, check answer, etc.)
3. **Self-Reflection**: Generates verbal feedback explaining what went wrong and what to try next

The loop:
```
Act -> Evaluate -> If fail: Reflect -> Store reflection -> Act again (with reflections)
```

**"Verbal reinforcement learning"**: The reflection IS the reward signal, but instead of a scalar value (like +1/-1 in RL), it's a natural language explanation. This is richer and more actionable.

### Why is this better than retry?
- Retry: "Task failed. Try again." (no information about what went wrong)
- Reflexion: "Task failed because I returned values instead of indices. The test expected [0, 1] but I returned [2, 7]. Next time I should return the indices i and j, not nums[i] and nums[j]." (specific, actionable feedback)

**Insider Tip:** Self-correction is real but limited. Research shows 2-3 reflection rounds are optimal -- beyond that, models start hallucinating reflections. In an interview, saying "self-correction works but has diminishing returns, and we need external verification for reliability" shows maturity.

In [ ]:
class ReflexionAgent:
    """
    Implementation of the Reflexion framework for coding tasks.
    
    Three components:
    - Actor: generates code solutions
    - Evaluator: runs test cases
    - Self-Reflection: generates verbal feedback on failures
    
    The agent stores reflections in memory and uses them to improve
    subsequent attempts -- no weight updates needed.
    """

    def __init__(self, model: str = "gpt-4o-mini", verbose: bool = True):
        self.model = model
        self.verbose = verbose
        self.reflection_memory: List[str] = []
        self.attempt_history: List[Dict] = []

    def act(self, task: CodingTask, memory: List[str]) -> str:
        """Generate a code solution using the task description + past reflections."""
        memory_text = ""
        if memory:
            memory_text = "\n\nLEARNINGS FROM PREVIOUS ATTEMPTS:\n"
            for i, ref in enumerate(memory):
                memory_text += f"\nAttempt {i+1} reflection:\n{ref}\n"

        prompt = f"""Write a Python function to solve this problem.

PROBLEM: {task.description}

Function signature: def {task.function_name}(...)
{memory_text}

Return ONLY the Python function code. No explanations, no test code, no markdown formatting."""

        if USE_LLM:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=500
            )
            code = response.choices[0].message.content.strip()
            # Clean up markdown fencing if present
            if code.startswith("```"):
                code = code.split("\n", 1)[1]
            if code.endswith("```"):
                code = code.rsplit("```", 1)[0]
            return code.strip()
        else:
            return self._simulate_act(task, memory)

    def evaluate(self, task: CodingTask, code: str) -> Dict[str, Any]:
        """Run the code against test cases."""
        return task.run_tests(code)

    def reflect(self, task: CodingTask, code: str, eval_result: Dict) -> str:
        """Generate verbal feedback explaining what went wrong."""
        error_text = "\n".join(eval_result["errors"])

        prompt = f"""You are a programming expert reflecting on a failed solution.

PROBLEM: {task.description}

MY SOLUTION:
{code}

TEST RESULTS:
Passed: {eval_result['passed']}, Failed: {eval_result['failed']}
Errors:
{error_text}

Reflect on what went wrong. Be specific:
1. What was the root cause of the failure?
2. What exactly should change in the next attempt?
3. What edge cases should I watch for?

Keep it concise (3-5 sentences)."""

        if USE_LLM:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=300
            )
            return response.choices[0].message.content.strip()
        else:
            return self._simulate_reflect(task, code, eval_result)

    def run(self, task: CodingTask, max_attempts: int = 3) -> Dict[str, Any]:
        """Full Reflexion loop: Act -> Evaluate -> Reflect -> repeat."""
        self.reflection_memory = []
        self.attempt_history = []

        for attempt in range(1, max_attempts + 1):
            if self.verbose:
                print(f"\n{'='*50}")
                print(f"Attempt {attempt}/{max_attempts} for {task.task_id}")
                print(f"{'='*50}")

            # ACT: generate solution
            code = self.act(task, self.reflection_memory)
            if self.verbose:
                print(f"\nGenerated code:\n{code}")

            # EVALUATE: run tests
            eval_result = self.evaluate(task, code)
            total_tests = eval_result["passed"] + eval_result["failed"]
            if self.verbose:
                print(f"\nTest results: {eval_result['passed']}/{total_tests} passed")

            self.attempt_history.append({
                "attempt": attempt,
                "code": code,
                "passed": eval_result["passed"],
                "total": total_tests,
                "errors": eval_result["errors"]
            })

            # Check if all tests pass
            if eval_result["failed"] == 0 and eval_result["passed"] > 0:
                if self.verbose:
                    print(f"\nAll tests passed on attempt {attempt}!")
                return {
                    "success": True,
                    "attempts": attempt,
                    "final_code": code,
                    "history": self.attempt_history
                }

            # REFLECT: generate verbal feedback
            if attempt < max_attempts:
                reflection = self.reflect(task, code, eval_result)
                self.reflection_memory.append(reflection)
                if self.verbose:
                    print(f"\nReflection: {reflection}")

        return {
            "success": False,
            "attempts": max_attempts,
            "final_code": code,
            "history": self.attempt_history
        }

    # --- Simulation methods (used when no LLM is available) ---

    def _simulate_act(self, task: CodingTask, memory: List[str]) -> str:
        """Simulated code generation that improves with reflections."""
        # Attempt 1: return a buggy version
        # Attempt 2+: if reflections mention the bug, return the fixed version

        correct_solutions = {
            "P1": '''def two_sum(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []
''',
            "P2": '''def is_palindrome(s):
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned == cleaned[::-1]
''',
            "P3": '''def max_profit(prices):
    if not prices:
        return 0
    min_price = prices[0]
    max_p = 0
    for price in prices[1:]:
        max_p = max(max_p, price - min_price)
        min_price = min(min_price, price)
    return max_p
''',
            "P4": '''def valid_parentheses(s):
    stack = []
    mapping = {")": "(", "]": "[", "}": "{"}
    for char in s:
        if char in mapping:
            top = stack.pop() if stack else '#'
            if mapping[char] != top:
                return False
        else:
            stack.append(char)
    return not stack
''',
            "P5": '''def fizzbuzz(n):
    result = []
    for i in range(1, n+1):
        if i % 15 == 0:
            result.append("FizzBuzz")
        elif i % 3 == 0:
            result.append("Fizz")
        elif i % 5 == 0:
            result.append("Buzz")
        else:
            result.append(str(i))
    return result
''',
        }

        buggy_solutions = {
            "P1": '''def two_sum(nums, target):
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            if nums[i] + nums[j] == target:
                return [nums[i], nums[j]]
    return []
''',
            "P2": '''def is_palindrome(s):
    cleaned = ''.join(c.lower() for c in s if c.isalpha())
    return cleaned == cleaned[::-1]
''',
            "P3": '''def max_profit(prices):
    max_p = 0
    for i in range(len(prices)):
        for j in range(len(prices)):
            max_p = max(max_p, prices[j] - prices[i])
    return max_p
''',
            "P4": '''def valid_parentheses(s):
    count = 0
    for c in s:
        if c in '([{':
            count += 1
        elif c in ')]}':
            count -= 1
        if count < 0:
            return False
    return count == 0
''',
            "P5": '''def fizzbuzz(n):
    result = []
    for i in range(1, n+1):
        if i % 3 == 0:
            result.append("Fizz")
        elif i % 5 == 0:
            result.append("Buzz")
        elif i % 15 == 0:
            result.append("FizzBuzz")
        else:
            result.append(str(i))
    return result
''',
        }

        # If we have reflections, use the correct solution
        # (simulates the agent learning from feedback)
        if memory:
            return correct_solutions.get(task.task_id, buggy_solutions.get(task.task_id, ""))
        else:
            return buggy_solutions.get(task.task_id, "")

    def _simulate_reflect(self, task: CodingTask, code: str, eval_result: Dict) -> str:
        """Simulated reflection."""
        reflections = {
            "P1": "I returned the VALUES at the indices instead of the INDICES themselves. The test expected [0, 1] (indices) but I returned [2, 7] (values). I need to return [i, j] instead of [nums[i], nums[j]]. I can also use a hashmap for O(n) time.",
            "P2": "I used isalpha() which only keeps letters, but I should use isalnum() to also keep digits. The problem says 'non-alphanumeric' so underscores should be removed but digits kept. The test 'ab_a' should become 'aba' which is a palindrome.",
            "P3": "My nested loop bug: j starts from 0 instead of i+1, so I'm considering selling before buying (j < i). This gives wrong results when prices only decrease. I should track min price seen so far and compute profit at each step for O(n) solution.",
            "P4": "Just counting parentheses doesn't work for matching pairs. '([)]' has equal open/close counts but is invalid. I need a STACK to match each closer with the most recent opener and check they're the same type.",
            "P5": "The order of checks matters! I check i%3 before i%15, so multiples of 15 get 'Fizz' instead of 'FizzBuzz'. The i%15 check must come FIRST, or I can use i%15==0 as the first condition.",
        }
        return reflections.get(task.task_id, "The solution has a bug. I need to re-examine the logic.")

print("ReflexionAgent defined with 3 components: act(), evaluate(), reflect()")
print("The run() method executes the full Reflexion loop.")

In [ ]:
# Run the Reflexion agent on a single task to demonstrate the framework

agent = ReflexionAgent(verbose=True)
result = agent.run(CODING_TASKS[0], max_attempts=3)  # two_sum

print(f"\n{'='*50}")
print(f"RESULT: {'SUCCESS' if result['success'] else 'FAILURE'}")
print(f"Solved in {result['attempts']} attempt(s)")
print(f"{'='*50}")

### Key Insight: Verbal Reinforcement Learning

Compare what happens with retry vs Reflexion:

| Aspect | Retry | Reflexion |
|--------|-------|-----------|
| **Feedback** | "Failed" (binary) | "Failed because I returned values instead of indices" (rich) |
| **Memory** | None | Stores reflections as text |
| **Weight updates** | None | None |
| **Improvement** | Random (hope for different sampling) | Directed (addresses specific failure) |
| **Cost** | N * base cost | N * (base + reflection) cost |

## 5. Self-Debugging: Reflexion for Code

Let's apply Reflexion to all 5 coding problems and measure the improvement over reflection rounds.

In [ ]:
class SelfDebuggingAgent(ReflexionAgent):
    """
    Extends ReflexionAgent with detailed tracking for code debugging.
    Tracks which tests pass/fail and what changed between attempts.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.detailed_history: List[Dict] = []

    def run_with_tracking(self, task: CodingTask, max_attempts: int = 3) -> Dict:
        """Run with detailed per-test tracking."""
        self.reflection_memory = []
        self.detailed_history = []

        for attempt in range(1, max_attempts + 1):
            code = self.act(task, self.reflection_memory)
            eval_result = self.evaluate(task, code)
            total = eval_result["passed"] + eval_result["failed"]

            self.detailed_history.append({
                "attempt": attempt,
                "code": code,
                "passed": eval_result["passed"],
                "total": total,
                "pass_rate": eval_result["passed"] / max(total, 1),
                "details": eval_result["details"]
            })

            if eval_result["failed"] == 0 and eval_result["passed"] > 0:
                return {
                    "success": True, "attempts": attempt,
                    "history": self.detailed_history
                }

            if attempt < max_attempts:
                reflection = self.reflect(task, code, eval_result)
                self.reflection_memory.append(reflection)

        return {
            "success": False, "attempts": max_attempts,
            "history": self.detailed_history
        }


# Run on all 5 coding problems
print("Running Self-Debugging Agent on all 5 coding problems...\n")

debug_agent = SelfDebuggingAgent(verbose=False)
all_results = {}

for task in CODING_TASKS:
    result = debug_agent.run_with_tracking(task, max_attempts=4)
    all_results[task.task_id] = result
    status = "PASS" if result["success"] else "FAIL"
    print(f"  {task.task_id} ({task.function_name}): {status} in {result['attempts']} attempt(s)")
    for h in result["history"]:
        print(f"    Attempt {h['attempt']}: {h['passed']}/{h['total']} tests passed ({h['pass_rate']:.0%})")

In [ ]:
# Measure: success by attempt 1 vs after 1 reflection vs after 3 reflections
# NOTE: sequential 'success by attempt k' (retries WITH feedback) is NOT canonical pass@k,
# which draws k INDEPENDENT samples and uses the unbiased Codex estimator (Chen et al., 2021).

# Collect pass rates at each attempt
max_attempts = 4
success_by_attempt = {k: 0 for k in range(1, max_attempts + 1)}
avg_pass_rate_at_k = {k: [] for k in range(1, max_attempts + 1)}

for task_id, result in all_results.items():
    for h in result["history"]:
        k = h["attempt"]
        avg_pass_rate_at_k[k].append(h["pass_rate"])
        if h["passed"] == h["total"] and h["total"] > 0:
            # This task was solved by attempt k
            for j in range(k, max_attempts + 1):
                success_by_attempt[j] += 1
            break

print("\n=== REFLEXION IMPROVEMENT ANALYSIS ===")
print(f"\nTotal tasks: {len(CODING_TASKS)}")
print(f"\n{'Metric':<35} {'Value':<10}")
print("-" * 45)
print(f"{'success by attempt 1 (no reflect)':<35} {success_by_attempt.get(1, 0)}/{len(CODING_TASKS)} = {success_by_attempt.get(1, 0)/len(CODING_TASKS):.0%}")
print(f"{'success by attempt 2 (1 reflection)':<35} {success_by_attempt.get(2, 0)}/{len(CODING_TASKS)} = {success_by_attempt.get(2, 0)/len(CODING_TASKS):.0%}")
print(f"{'success by attempt 3 (2 reflections)':<35} {success_by_attempt.get(3, 0)}/{len(CODING_TASKS)} = {success_by_attempt.get(3, 0)/len(CODING_TASKS):.0%}")
print(f"{'success by attempt 4 (3 reflections)':<35} {success_by_attempt.get(4, 0)}/{len(CODING_TASKS)} = {success_by_attempt.get(4, 0)/len(CODING_TASKS):.0%}")

print(f"\nAverage test pass rate by attempt:")
for k in range(1, max_attempts + 1):
    rates = avg_pass_rate_at_k[k]
    if rates:
        print(f"  Attempt {k}: {np.mean(rates):.0%} avg test pass rate")

In [ ]:
# Visualization: improvement over reflection rounds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: success-by-attempt-k improvement
ax = axes[0]
ks = list(range(1, max_attempts + 1))
pass_rates = [success_by_attempt.get(k, 0) / len(CODING_TASKS) for k in ks]
labels = ["by attempt 1\n(no reflection)", "by attempt 2\n(1 reflection)",
          "by attempt 3\n(2 reflections)", "by attempt 4\n(3 reflections)"]

bars = ax.bar(ks, pass_rates, color=['#e74c3c', '#f39c12', '#2ecc71', '#27ae60'],
              edgecolor='black', linewidth=0.8)
for bar, rate in zip(bars, pass_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{rate:.0%}", ha='center', fontsize=12, fontweight='bold')
ax.set_xticks(ks)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Task Solve Rate', fontsize=11)
ax.set_title('Reflexion: Improvement Over Attempts', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: per-task pass rate trajectory
ax2 = axes[1]
colors_map = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for idx, (task_id, result) in enumerate(all_results.items()):
    attempts = [h["attempt"] for h in result["history"]]
    rates = [h["pass_rate"] for h in result["history"]]
    task = [t for t in CODING_TASKS if t.task_id == task_id][0]
    ax2.plot(attempts, rates, 'o-', color=colors_map[idx],
             label=f"{task_id}: {task.function_name}", markersize=8, linewidth=2)

ax2.set_xlabel('Attempt Number', fontsize=11)
ax2.set_ylabel('Test Pass Rate', fontsize=11)
ax2.set_title('Per-Task Improvement Trajectory', fontsize=13, fontweight='bold')
ax2.set_ylim(-0.05, 1.15)
ax2.set_xticks(range(1, max_attempts + 1))
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("  - Reflection provides the biggest gain from attempt 1 -> 2")
print("  - Subsequent reflections have diminishing returns")
print("  - Some tasks are solved immediately; reflection helps the harder ones")

## 6. Constitutional Self-Critique

Constitutional AI (Bai et al., 2022) introduced the idea of self-critique against principles. We can apply this to agents: after each action, the agent critiques itself against a set of constitutional principles.

**Connection to safety-critical TTS work**: Verification loops in a safety-critical text-to-speech (TTS) project (checking that generated speech matches the script verbatim) are a domain-specific form of self-correction. The "constitution" there is: "The output must match the input exactly."

In [ ]:
class ConstitutionalAgent:
    """
    An agent that critiques its own outputs against a set of principles
    before presenting them to the user.
    
    Principles are natural language rules that the agent checks its
    output against. If any principle is violated, the agent revises.
    """

    def __init__(self, principles: List[str], model: str = "gpt-4o-mini", verbose: bool = True):
        self.principles = principles
        self.model = model
        self.verbose = verbose

    def generate(self, task: str) -> str:
        """Generate an initial response to the task."""
        if USE_LLM:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": task}],
                temperature=0.7,
                max_tokens=500
            )
            return response.choices[0].message.content.strip()
        else:
            return self._simulate_generate(task)

    def critique(self, task: str, response: str) -> Dict[str, Any]:
        """Critique the response against each principle."""
        critiques = []
        any_violation = False

        for principle in self.principles:
            if USE_LLM:
                prompt = f"""Given this task and response, does the response violate this principle?

Task: {task}
Response: {response}
Principle: {principle}

Answer with: PASS (no violation) or FAIL (violation detected) followed by a brief explanation."""
                result = client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.1,
                    max_tokens=150
                )
                critique_text = result.choices[0].message.content.strip()
                violated = critique_text.upper().startswith("FAIL")
            else:
                violated, critique_text = self._simulate_critique(task, response, principle)

            critiques.append({
                "principle": principle,
                "violated": violated,
                "critique": critique_text
            })
            if violated:
                any_violation = True

        return {"critiques": critiques, "any_violation": any_violation}

    def revise(self, task: str, response: str, critiques: List[Dict]) -> str:
        """Revise the response based on critique feedback."""
        violations = [c for c in critiques if c["violated"]]
        if not violations:
            return response

        feedback = "\n".join(
            f"- Principle '{c['principle']}': {c['critique']}" for c in violations
        )

        if USE_LLM:
            prompt = f"""Revise this response to fix the identified violations.

Task: {task}
Original response: {response}

Violations found:
{feedback}

Provide the revised response only."""
            result = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )
            return result.choices[0].message.content.strip()
        else:
            return self._simulate_revise(task, response, violations)

    def run(self, task: str, max_revisions: int = 2) -> Dict[str, Any]:
        """Generate, critique, and revise until all principles are satisfied."""
        history = []

        # Initial generation
        response = self.generate(task)
        if self.verbose:
            print(f"Initial response:\n  {response[:200]}...\n")

        for revision in range(max_revisions + 1):
            # Critique
            critique_result = self.critique(task, response)

            history.append({
                "revision": revision,
                "response": response,
                "critiques": critique_result["critiques"],
                "has_violations": critique_result["any_violation"]
            })

            if self.verbose:
                for c in critique_result["critiques"]:
                    status = "FAIL" if c["violated"] else "PASS"
                    print(f"  [{status}] {c['principle']}: {c['critique'][:80]}")

            if not critique_result["any_violation"]:
                if self.verbose:
                    print(f"\n  All principles satisfied after {revision} revision(s).")
                break

            if revision < max_revisions:
                if self.verbose:
                    print(f"\n  Revising...")
                response = self.revise(task, response, critique_result["critiques"])
                if self.verbose:
                    print(f"  Revised response:\n  {response[:200]}...\n")

        return {
            "final_response": response,
            "revisions": len(history) - 1,
            "all_satisfied": not history[-1]["has_violations"],
            "history": history
        }

    # --- Simulation methods ---

    def _simulate_generate(self, task: str) -> str:
        responses = {
            "recommendation": "You should definitely invest all your savings in cryptocurrency. Bitcoin is guaranteed to go up. Also, I can share your personal financial data with third-party advisors to get better recommendations.",
            "email": "Subject: URGENT - Account Compromised\n\nDear user, your account has been hacked. Click this link immediately to reset your password: http://totally-legit-link.com. We need your SSN to verify.",
        }
        for key, resp in responses.items():
            if key in task.lower():
                return resp
        return "Here is a helpful response to your request. I'll do my best to assist you with accurate and safe information."

    def _simulate_critique(self, task: str, response: str, principle: str) -> Tuple[bool, str]:
        """Simulate principle checking."""
        response_lower = response.lower()

        if "safe" in principle.lower() or "harm" in principle.lower():
            if any(w in response_lower for w in ["guaranteed", "click this link", "hacked", "ssn"]):
                return True, "FAIL: Response contains potentially harmful advice or deceptive content."
            return False, "PASS: Response appears safe and non-harmful."

        if "honest" in principle.lower() or "accurate" in principle.lower():
            if "guaranteed" in response_lower:
                return True, "FAIL: Makes unsubstantiated guarantees about investment returns."
            return False, "PASS: Response appears honest and factual."

        if "privacy" in principle.lower() or "personal" in principle.lower():
            if any(w in response_lower for w in ["share your personal", "ssn", "personal data"]):
                return True, "FAIL: Response requests or shares private information inappropriately."
            return False, "PASS: Response respects user privacy."

        return False, "PASS: No violations detected for this principle."

    def _simulate_revise(self, task: str, response: str, violations: List[Dict]) -> str:
        """Simulate revision to fix violations."""
        if "recommendation" in task.lower() or "invest" in task.lower():
            return "I'd recommend diversifying your investments across different asset classes. Consider consulting with a licensed financial advisor who can review your specific situation. Past performance doesn't guarantee future results, so please be cautious with any investment."
        elif "email" in task.lower():
            return "Subject: Account Security Notice\n\nDear user, we noticed unusual activity on your account. Please log in directly through our official website (not via email links) to review your recent activity. Never share your password or personal information via email. Contact our support team if you need assistance."
        return response


print("ConstitutionalAgent defined.")
print("Uses generate -> critique -> revise loop against natural language principles.")

In [ ]:
# Demo: Constitutional self-critique on a risky task

principles = [
    "The response must be safe and not cause harm to the user.",
    "The response must be honest and not make unsubstantiated claims.",
    "The response must respect user privacy and not request unnecessary personal information.",
]

const_agent = ConstitutionalAgent(principles=principles, verbose=True)

print("="*60)
print("Constitutional Self-Critique Demo")
print("="*60)

task = "Write a financial recommendation for a user asking about investment."
print(f"\nTask: {task}\n")

result = const_agent.run(task, max_revisions=2)

print(f"\n{'='*60}")
print(f"Final response after {result['revisions']} revision(s):")
print(f"  {result['final_response']}")
print(f"\nAll principles satisfied: {result['all_satisfied']}")

### Connection to Safety-Critical TTS

The Constitutional self-critique pattern is structurally identical to the verification loop in a safety-critical text-to-speech (TTS) project:

| Safety-critical TTS | Constitutional Agent |
|-------------|---------------------|
| Generate speech | Generate response |
| Compare to script (verification) | Critique against principles |
| If mismatch: regenerate | If violation: revise |
| Constitution: "must match script exactly" | Constitution: safety, honesty, privacy |

The key insight is the same: **verification is cheaper than generation**. It's easier to check if a response is safe than to generate a guaranteed-safe response on the first try.

## 7. When Does Self-Correction Fail?

Self-correction is not a silver bullet. Understanding its failure modes is critical for interviews.

In [ ]:
# Simulate reflection quality over rounds to show diminishing returns

def simulate_reflection_quality(
    n_tasks: int = 20,
    max_reflections: int = 6,
    base_solve_rate: float = 0.4,
    reflection_boost: float = 0.20,
    diminishing_factor: float = 0.55,
    overcorrection_rate: float = 0.08
) -> Dict[str, List]:
    """
    Simulate how reflection quality changes over rounds.
    
    Key phenomena:
    - Diminishing returns: each reflection helps less
    - Over-correction: sometimes reflection makes things worse
    - Plateau: after N reflections, no more improvement
    """
    np.random.seed(42)
    results = {"round": [], "avg_pass_rate": [], "newly_solved": [],
               "overcorrected": [], "cumulative_solved": []}

    # Track each task's state
    task_states = []
    for i in range(n_tasks):
        # Each task has a difficulty level
        difficulty = np.random.uniform(0.1, 0.9)
        task_states.append({
            "difficulty": difficulty,
            "solved": False,
            "pass_rate": max(0, min(1, base_solve_rate + np.random.normal(0, 0.15) - difficulty * 0.3)),
            "was_solved_last_round": False
        })

    cumulative_solved = 0

    for round_num in range(max_reflections + 1):
        newly_solved = 0
        overcorrected = 0
        pass_rates = []

        for task in task_states:
            if round_num > 0:
                # Apply reflection boost (diminishing)
                boost = reflection_boost * (diminishing_factor ** (round_num - 1))
                task["pass_rate"] = min(1.0, task["pass_rate"] + boost * (1 - task["difficulty"]))

                # Over-correction: sometimes reflection makes it worse
                if np.random.random() < overcorrection_rate:
                    task["pass_rate"] = max(0, task["pass_rate"] - 0.2)
                    if task["was_solved_last_round"]:
                        task["solved"] = False
                        overcorrected += 1
                        cumulative_solved -= 1

            # Check if newly solved
            task["was_solved_last_round"] = task["solved"]
            if task["pass_rate"] >= 0.99 and not task["solved"]:
                task["solved"] = True
                newly_solved += 1
                cumulative_solved += 1

            pass_rates.append(task["pass_rate"])

        results["round"].append(round_num)
        results["avg_pass_rate"].append(np.mean(pass_rates))
        results["newly_solved"].append(newly_solved)
        results["overcorrected"].append(overcorrected)
        results["cumulative_solved"].append(cumulative_solved)

    return results


sim_results = simulate_reflection_quality()

print("Simulation of reflection quality over rounds:\n")
print(f"{'Round':<8} {'Avg Pass Rate':<15} {'Newly Solved':<14} {'Overcorrected':<15} {'Cumulative Solved':<18}")
print("-" * 70)
for i in range(len(sim_results["round"])):
    print(f"{sim_results['round'][i]:<8} "
          f"{sim_results['avg_pass_rate'][i]:<15.1%} "
          f"{sim_results['newly_solved'][i]:<14} "
          f"{sim_results['overcorrected'][i]:<15} "
          f"{sim_results['cumulative_solved'][i]:<18}")

In [ ]:
# Visualize diminishing returns and failure modes

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

rounds = sim_results["round"]

# Plot 1: Average pass rate over rounds (diminishing returns curve)
ax = axes[0]
ax.plot(rounds, sim_results["avg_pass_rate"], 'bo-', linewidth=2, markersize=8)
ax.fill_between(rounds, sim_results["avg_pass_rate"], alpha=0.15, color='blue')
ax.axhline(y=sim_results["avg_pass_rate"][-1], color='gray', linestyle='--', alpha=0.5, label='Plateau')
ax.set_xlabel('Reflection Round', fontsize=11)
ax.set_ylabel('Average Test Pass Rate', fontsize=11)
ax.set_title('Diminishing Returns of Reflection', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate the sweet spot
improvements = [0] + [sim_results["avg_pass_rate"][i] - sim_results["avg_pass_rate"][i-1]
                       for i in range(1, len(rounds))]
sweet_spot = np.argmax([imp if imp > 0.01 else 0 for imp in improvements])
ax.annotate(f'Best gain: round {sweet_spot}',
            xy=(sweet_spot, sim_results["avg_pass_rate"][sweet_spot]),
            xytext=(sweet_spot + 1, sim_results["avg_pass_rate"][sweet_spot] - 0.1),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')

# Plot 2: Marginal improvement per round
ax2 = axes[1]
marginal = [0] + [sim_results["avg_pass_rate"][i] - sim_results["avg_pass_rate"][i-1]
                   for i in range(1, len(rounds))]
colors = ['green' if m > 0 else 'red' for m in marginal]
ax2.bar(rounds, marginal, color=colors, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xlabel('Reflection Round', fontsize=11)
ax2.set_ylabel('Marginal Improvement', fontsize=11)
ax2.set_title('Marginal Gain Per Reflection', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add the "diminishing returns" annotation
ax2.annotate('Diminishing\nreturns', xy=(3, marginal[3] if len(marginal) > 3 else 0),
             xytext=(4.5, max(marginal) * 0.7),
             arrowprops=dict(arrowstyle='->', color='gray'),
             fontsize=9, color='gray')

# Plot 3: Cumulative solved with overcorrection annotations
ax3 = axes[2]
ax3.plot(rounds, sim_results["cumulative_solved"], 'go-', linewidth=2, markersize=8, label='Cumulative solved')
ax3.bar(rounds, sim_results["overcorrected"], color='red', alpha=0.5, label='Overcorrected (regressed)')
ax3.set_xlabel('Reflection Round', fontsize=11)
ax3.set_ylabel('Number of Tasks', fontsize=11)
ax3.set_title('Solved Tasks & Over-correction', fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("FAILURE MODES OF SELF-CORRECTION")
print("="*60)
print()
print("1. DIMINISHING RETURNS:")
print("   After 2-3 reflections, improvements plateau.")
print("   The easy-to-fix bugs get fixed first; harder bugs persist.")
print()
print("2. HALLUCINATED REFLECTIONS:")
print("   Agent generates plausible but WRONG self-feedback.")
print("   Example: 'I failed because the input was malformed'")
print("   when actually the algorithm was wrong.")
print()
print("3. OVER-CORRECTION:")
print("   Agent changes things that were already correct.")
print("   Example: fixes bug A but introduces bug B in the process.")
print()
print("RULE OF THUMB: 2-3 reflection rounds is optimal for most tasks.")
print("Beyond that, the cost of additional reflections outweighs the gain.")

### Detailed Failure Mode Analysis

| Failure Mode | What Happens | When It Occurs | Mitigation |
|-------------|-------------|----------------|------------|
| **Diminishing Returns** | Each reflection helps less | After 2-3 rounds | Cap reflections; switch to different strategy |
| **Hallucinated Reflections** | Wrong diagnosis of failure | When error message is ambiguous | Use structured error analysis; external verifier |
| **Over-correction** | Fixes one bug, introduces another | Complex multi-component code | Track ALL test results, not just failing ones |
| **Reflection Loop** | Agent keeps reflecting but never converges | Fundamental misunderstanding of task | Add diversity; use different prompts per attempt |
| **Sycophantic Reflection** | Agent agrees with its own bad solution | Weak evaluator signal | Use external evaluator (tests, not self-judgment) |

## 8. Comparing Strategies

Let's compare three approaches on the same tasks:
1. **No reflection**: Just generate once (pass@1)
2. **Reflexion**: Generate, reflect, regenerate
3. **Majority voting**: Generate N solutions, pick the most common answer

In [ ]:
def simulate_strategy_comparison(
    n_tasks: int = 30,
    seed: int = 42
) -> Dict[str, Dict]:
    """
    Compare three strategies on the same tasks.
    Simulates realistic pass rates and costs.
    """
    np.random.seed(seed)

    # Generate task difficulties
    difficulties = np.random.uniform(0.1, 0.95, n_tasks)

    strategies = {}

    # Strategy 1: No reflection (pass@1)
    base_rate = 0.6
    solved_no_reflect = []
    for d in difficulties:
        p_solve = max(0, min(1, base_rate - d * 0.4 + np.random.normal(0, 0.1)))
        solved_no_reflect.append(np.random.random() < p_solve)

    strategies["No Reflection\n(pass@1)"] = {
        "solve_rate": np.mean(solved_no_reflect),
        "avg_cost_tokens": 500,  # 1 generation
        "total_cost": 500 * n_tasks,
        "solved": solved_no_reflect
    }

    # Strategy 2: Reflexion (3 attempts)
    solved_reflexion = []
    reflexion_costs = []
    for i, d in enumerate(difficulties):
        p_solve = max(0, min(1, base_rate - d * 0.4 + np.random.normal(0, 0.1)))
        solved = False
        cost = 500  # initial generation
        for attempt in range(3):
            if np.random.random() < p_solve:
                solved = True
                break
            # Reflection boosts probability
            p_solve = min(1, p_solve + 0.15 * (0.6 ** attempt))
            cost += 300  # reflection cost
            cost += 500  # re-generation cost
        solved_reflexion.append(solved)
        reflexion_costs.append(cost)

    strategies["Reflexion\n(3 attempts)"] = {
        "solve_rate": np.mean(solved_reflexion),
        "avg_cost_tokens": np.mean(reflexion_costs),
        "total_cost": sum(reflexion_costs),
        "solved": solved_reflexion
    }

    # Strategy 3: Majority voting (5 samples)
    solved_majority = []
    majority_costs = []
    n_samples = 5
    for i, d in enumerate(difficulties):
        p_solve = max(0, min(1, base_rate - d * 0.4 + np.random.normal(0, 0.1)))
        # Generate N solutions, take majority vote
        votes = [np.random.random() < p_solve for _ in range(n_samples)]
        solved = sum(votes) > n_samples // 2  # majority correct
        solved_majority.append(solved)
        majority_costs.append(500 * n_samples)  # N generations

    strategies["Majority Vote\n(5 samples)"] = {
        "solve_rate": np.mean(solved_majority),
        "avg_cost_tokens": np.mean(majority_costs),
        "total_cost": sum(majority_costs),
        "solved": solved_majority
    }

    return strategies


strategies = simulate_strategy_comparison()

print("=" * 65)
print("STRATEGY COMPARISON (30 tasks)")
print("=" * 65)
print(f"\n{'Strategy':<25} {'Solve Rate':<14} {'Avg Tokens':<14} {'Total Cost':<12}")
print("-" * 65)
for name, data in strategies.items():
    clean_name = name.replace('\n', ' ')
    print(f"{clean_name:<25} {data['solve_rate']:<14.1%} {data['avg_cost_tokens']:<14,.0f} {data['total_cost']:<12,}")

In [ ]:
# Visualize strategy comparison

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

names = list(strategies.keys())
solve_rates = [strategies[n]["solve_rate"] for n in names]
avg_costs = [strategies[n]["avg_cost_tokens"] for n in names]
total_costs = [strategies[n]["total_cost"] for n in names]

colors = ['#3498db', '#2ecc71', '#e74c3c']

# Plot 1: Solve rates
ax = axes[0]
bars = ax.bar(range(len(names)), solve_rates, color=colors, edgecolor='black', linewidth=0.8)
for bar, rate in zip(bars, solve_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{rate:.0%}", ha='center', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Task Solve Rate', fontsize=11)
ax.set_title('Accuracy Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Average cost per task
ax2 = axes[1]
bars2 = ax2.bar(range(len(names)), avg_costs, color=colors, edgecolor='black', linewidth=0.8)
for bar, cost in zip(bars2, avg_costs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{cost:,.0f}", ha='center', fontsize=10)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(names, fontsize=8)
ax2.set_ylabel('Avg Tokens per Task', fontsize=11)
ax2.set_title('Cost Comparison', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Efficiency (solve rate per 1000 tokens)
ax3 = axes[2]
efficiency = [sr / (ac / 1000) for sr, ac in zip(solve_rates, avg_costs)]
bars3 = ax3.bar(range(len(names)), efficiency, color=colors, edgecolor='black', linewidth=0.8)
for bar, eff in zip(bars3, efficiency):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{eff:.2f}", ha='center', fontsize=10)
ax3.set_xticks(range(len(names)))
ax3.set_xticklabels(names, fontsize=8)
ax3.set_ylabel('Solve Rate per 1K Tokens', fontsize=11)
ax3.set_title('Efficiency (Quality/Cost)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("  - Reflexion achieves higher solve rates than no-reflection")
print("  - Majority voting can match or beat Reflexion but costs more")
print("  - Reflexion is more TOKEN-EFFICIENT (better quality per token spent)")
print("  - The best strategy depends on your budget constraint")
print("\nWhen to use each:")
print("  - No reflection: Fast responses, budget-constrained, easy tasks")
print("  - Reflexion: Medium budget, tasks with clear error signals (tests, verification)")
print("  - Majority vote: High budget, tasks without clear error signals (open-ended generation)")

## 9. "Why Does This Work?"

### Why is verbal feedback better than scalar reward?

In traditional RL, the reward is a number: +1 for success, -1 for failure. This tells the agent WHAT happened but not WHY.

Verbal feedback is richer:
- Scalar: "Task failed (-1)"
- Verbal: "I failed because I returned values instead of indices. The test expected [0, 1] but I returned [2, 7]. Next time, I should return [i, j] instead of [nums[i], nums[j]]."

The verbal signal is **more actionable** because it encodes the causal structure of the failure. The agent doesn't need to explore blindly -- it knows exactly what to change.

### How is Reflexion different from RLHF?

| Aspect | RLHF | Reflexion |
|--------|------|-----------|
| **When** | Training time | Inference time |
| **What changes** | Model weights (permanent) | In-context memory (temporary) |
| **Feedback** | Human preference labels | Self-generated verbal reflections |
| **Cost** | Very expensive (human labor + training) | Cheap (just extra API calls) |
| **Persistence** | Persists across all future tasks | Resets each episode (unless you save reflections) |
| **Scalability** | Scales with training data | Scales with inference compute |

**Key insight**: Reflexion is "learning" without weight updates. It's in-context learning from self-generated feedback. This means it's fast to deploy but doesn't persist -- each new task starts fresh (unless you maintain a reflection database).

### When should you use reflection vs fine-tuning?

**Use reflection when:**
- You need to improve performance without access to model weights
- Tasks are diverse (each task is different)
- You have a good evaluator (tests, ground truth, verifier)
- Latency is acceptable (2-3x slower for reflection rounds)

**Use fine-tuning when:**
- You see the same failure patterns repeatedly across many tasks
- You want permanent improvement
- You have labeled training data
- Inference cost/latency is critical (fine-tuning is one-time cost)

**Use both when:**
- Fine-tune on common failure patterns, then use reflection for edge cases at inference time. This is the production sweet spot.

---
## Interview Question Bank

*Reflection questions test whether you understand the limits of self-assessment. The key insight interviewers want: self-correction is powerful for verifiable domains but dangerous when the model cannot reliably judge its own output.*

---

### Q1: "When does self-correction work and when does it fail?"

**What this tests**: Nuanced understanding of reflection. Most candidates are either overly optimistic ("just ask the model to check its work") or overly pessimistic ("models cannot self-correct at all").

**Good answer** (hire):
- Works for verifiable errors: code that does not compile, math that does not add up, factual claims that can be checked against a source
- Fails for subjective quality: writing style, design decisions, creative work

**Great answer** (strong hire):
- **Works when there is an external signal**: Compilation errors, test failures, API error codes, type checking -- any feedback that does not come from the model itself
- **Works for format/structure**: "Your output should be JSON but you produced plain text" -- the model can reliably detect and fix structural errors
- **Fails for knowledge gaps**: If the model does not know the correct answer, reflection will not magically provide it. "Hallucinated reflections" are a real phenomenon -- the model confidently "corrects" a right answer to a wrong one
- **Fails for subtle errors**: Off-by-one bugs, edge cases, security vulnerabilities -- these require domain expertise that reflection alone does not provide
- **Diminishing returns**: First reflection round catches 60-70% of fixable errors. Second round catches maybe 10-15% more. Third round and beyond is noise. The cost of additional rounds rarely justifies the marginal improvement.
- **The self-evaluation problem**: Models are poorly calibrated about their own correctness. They rate their outputs as high-quality even when they are wrong. External verification (tests, linters, humans) is far more reliable than self-assessment.

**Red flag**: Claims self-correction "always works" or "never works." Does not distinguish between verifiable and unverifiable domains.

**Follow-up**: "Design a system that combines self-correction with external verification"
- Good: run code, check output against expected results, then reflect
- Great: multi-layer verification pipeline: (1) self-check (fast, cheap, catches obvious errors), (2) automated verification (tests, linters, type checkers), (3) LLM-as-judge with a different model (catches subtle errors the author-model might miss), (4) human review for high-stakes outputs. Each layer is a gate -- only items that pass the cheaper check proceed to the more expensive one.

---

### Q2: "Reflexion vs fine-tuning -- when would you use each?"

**What this tests**: Understanding of the inference-time vs training-time trade-off. This connects directly to the Sprint 2 post-training material.

**Good answer**:
- Reflexion is inference-time: no training needed, works immediately, but costs extra tokens per query
- Fine-tuning is training-time: requires data and compute upfront, but cheaper per query after training

**Great answer**: Provides a decision framework:

| Factor | Use Reflexion | Use Fine-tuning |
|--------|--------------|-----------------|
| **Error frequency** | Rare errors (not worth retraining) | Systematic errors (same mistake on many inputs) |
| **Error type** | Diverse, unpredictable | Consistent, patterned |
| **Cost structure** | Few queries, high value each | Many queries, cost per query matters |
| **Time to fix** | Need a fix NOW (no training time) | Can wait days-weeks for training |
| **Data availability** | No training data available | Have examples of correct behavior |
| **Flexibility** | Need to handle novel error types | Error types are well-understood |

**The cost analysis**:
- Reflexion: adds 1-3 extra LLM calls per task. At $10/1M tokens, that is $0.01-0.05 per task.
- Fine-tuning: $100-10,000 upfront, but then each query is cheaper (fine-tuned small model vs large model + reflection)
- Break-even: if you have >10,000 tasks with the same error pattern, fine-tuning wins. Below that, Reflexion wins.

**The distillation approach** (best of both worlds): Use Reflexion to generate corrected outputs, then fine-tune on the (input, corrected_output) pairs. This is how many production systems improve over time.

**Red flag**: Does not consider cost. Cannot explain when one approach is strictly better than the other.

---
## Production Implementation Notes

*How self-correction works in real deployed systems.*

### Reflection in Production Systems

**Claude Code (Anthropic)**:
- Uses a "verify then commit" pattern: generates code, runs it, checks output, re-generates if tests fail
- The reflection is grounded in external signals (test results, linter output), not self-assessment
- Retry budgets are configurable and implementation-specific (no fixed public number); when the budget is exhausted, the agent escalates to the user

**GitHub Copilot Workspace** (sunset in 2025; folded into the GitHub Copilot coding agent):
- Multi-step plan generation with user review between steps
- Self-correction is implicit: each step's output is validated before proceeding to the next
- Human-in-the-loop acts as the external verifier

**Devin / SWE agents**:
- Run code in a sandbox, observe errors, fix and re-run
- The key insight: the sandbox IS the verifier. Self-correction works because the feedback is objective (tests pass or fail)
- Without a sandbox, self-correction for code is unreliable

### When to Use Each Strategy in Production

```
Decision tree:
  Is the output verifiable? (code, math, structured data)
    YES -> Self-correction with external verification (run it, check it, fix it)
    NO  -> Is it high-stakes? (medical, legal, financial)
      YES -> Human review (do not trust self-correction)
      NO  -> LLM-as-judge with a different model (cheap, reasonably reliable)
        Still unsure? -> Sample multiple outputs, pick the best (best-of-N)
```

### Cost of Reflection

| Strategy | Extra Cost | Reliability Gain | When Worth It |
|----------|-----------|-----------------|---------------|
| **1 reflection round** | 2x base cost | +15-25% accuracy | Almost always |
| **2 reflection rounds** | 3x base cost | +5-10% more | High-value tasks only |
| **3+ reflection rounds** | 4x+ base cost | +1-3% more | Rarely justified |
| **Best-of-N (N=5)** | 5x base cost | +10-20% | When reflection cannot help (no verifier) |
| **External verifier** | Verifier cost (varies) | +20-40% | When available, always |

### The Uncomfortable Truth About Self-Correction

> A recent study (Huang et al., 2023, "Large Language Models Cannot Self-Correct Reasoning Yet") showed that without external feedback, LLMs often make their answers *worse* through self-correction. The model second-guesses correct answers more than it fixes incorrect ones. This is a critical finding for anyone building production systems: **never rely on pure self-assessment. Always ground reflection in external signals.**

---
## How This Gets Tested in Interviews

### The Reflection Trap

Reflection questions are traps for candidates who have read too many papers and not built enough systems. The trap works like this:

**Interviewer**: "How would you make your agent more reliable?"
**Weak candidate**: "I would add self-reflection. The agent critiques its own output and improves it."
**Interviewer**: "What if the reflection is wrong?"
**Weak candidate**: "...add another round of reflection?"

The interviewer is testing whether you understand that **self-correction without external grounding is circular reasoning**. The model that made the mistake is the same model evaluating the mistake. If it knew the right answer, it would have generated it the first time.

### The Correct Framework

When asked about reflection/self-correction, immediately establish:

1. **What is the verification signal?** (Tests? Linter? Type checker? Human? Different model?)
2. **Is the domain verifiable or subjective?** (Code: verifiable. Creative writing: subjective.)
3. **What is the cost/benefit?** (One reflection round is usually worth it. Three rounds rarely are.)
4. **What is the fallback?** (When reflection fails, what happens? Escalate to human? Return uncertainty?)

### How This Connects to Your Research

In a research discussion, connect self-correction to:
- **RL**: Reflexion is verbal RL -- the reflection serves as a "reward signal" expressed in natural language. But unlike RL, there is no gradient update, so the "learning" only persists within the episode.
- **Debate (Irving et al.)**: Using two models to check each other is more reliable than self-check, because adversarial dynamics surface errors that self-assessment misses
- **Constitutional AI**: Anthropic's approach uses the model to critique its own outputs against a set of principles -- but the principles are externally defined, which provides the grounding signal
- **Process Reward Models**: training a verifier model to evaluate each step of reasoning, rather than just the final answer. More reliable than self-assessment because the verifier is trained specifically to detect errors.

### The One-Sentence Answer

If you need to summarize self-correction in one sentence for an interview:

> "Self-correction works when grounded in external verification and fails when the model is its own judge -- the key is designing systems where the feedback signal comes from outside the model."

## 10. Flashcard Summary

Study these for interview prep. Cover the answer, try to recall, then check.

---

**Q1**: What are the three components of the Reflexion framework?

**A1**: (1) Actor -- generates actions/solutions, (2) Evaluator -- scores the outcome, (3) Self-Reflection -- generates verbal feedback explaining what went wrong. The loop: Act -> Evaluate -> Reflect -> Store reflection -> Act again.

---

**Q2**: What is verbal reinforcement learning?

**A2**: Instead of scalar rewards (+1/-1), the reward signal is natural language reflecting on what went wrong and what to try next. This is richer and more actionable than scalar feedback because it encodes the causal structure of failures.

---

**Q3**: How does Reflexion differ from simple retry?

**A3**: Retry runs the same agent again with no feedback -- it relies on random sampling to get a different result. Reflexion generates specific, verbal feedback about WHY the attempt failed and stores it in memory, so the next attempt is informed and directed.

---

**Q4**: What are three failure modes of self-correction?

**A4**: (1) Diminishing returns -- improvements plateau after 2-3 rounds. (2) Hallucinated reflections -- agent generates plausible but wrong self-feedback. (3) Over-correction -- agent fixes one issue but breaks something that was working.

---

**Q5**: What is the optimal number of reflection rounds?

**A5**: 2-3 rounds for most tasks. The first reflection captures the biggest gain. After 3 rounds, marginal improvement is minimal and over-correction risk increases.

---

**Q6**: How is Reflexion different from RLHF?

**A6**: RLHF updates model weights permanently using human feedback at training time. Reflexion uses self-generated verbal feedback at inference time with no weight updates -- it's purely in-context learning.

---

**Q7**: What is Constitutional Self-Critique?

**A7**: After generating a response, the agent critiques it against a set of natural language principles (safety, honesty, privacy). If any principle is violated, the agent revises before presenting the response. Generate -> Critique -> Revise.

---

**Q8**: When should you use reflection vs fine-tuning?

**A8**: Reflection: inference-time, diverse tasks, no access to weights, good evaluator available. Fine-tuning: persistent improvement, repeated failure patterns, training data available. Best practice: fine-tune on common patterns, reflect on edge cases.

---

**Q9**: How does Reflexion compare to majority voting?

**A9**: Reflexion is more token-efficient (directed improvement vs random sampling). Majority voting works better when there's no clear error signal. Reflexion works better when you have a verifier (tests, ground truth). Cost: Reflexion ~2-3x base, majority voting ~5x base.

---

**Q10**: Why does verification tend to be easier than generation?

**A10**: Verification (checking if output is correct) is typically a simpler problem than generation (producing a correct output). This asymmetry makes self-correction valuable: generate an imperfect solution, then use the easier verification step to catch and fix errors.

---

**Q11**: What is self-debugging for code?

**A11**: A specific application of Reflexion to coding: generate code, run tests, if tests fail analyze the error message and failed test case, reflect on the bug, generate an improved solution. The test suite serves as the evaluator.

---

**Q12**: What is the connection between safety-critical TTS verification and self-correction?

**A12**: Both use the same pattern: generate -> verify -> if wrong, regenerate with feedback. In a safety-critical TTS system, the "constitution" is exact script match. The key insight is that verification (checking speech matches script) is cheaper than generation (producing correct speech on first try).

## 11. Paper Guide

### Reflexion: Language Agents with Verbal Reinforcement Learning
**Authors**: Noah Shinn, Federico Cassano, Ashwin Gopinath, Karthik Narasimhan, Shunyu Yao (2023)

**Link**: [https://arxiv.org/abs/2303.11366](https://arxiv.org/abs/2303.11366)

### Key Contributions:
1. **Verbal reinforcement learning**: Uses natural language reflections instead of scalar rewards
2. **No weight updates needed**: All "learning" happens in-context through reflection memory
3. **Significant improvements**: +10-20% on coding (HumanEval), decision-making (ALFWorld), and reasoning (HotPotQA) tasks

### Architecture:
- **Actor (M_a)**: The base LLM that generates actions
- **Evaluator (M_e)**: Provides binary or scalar feedback (tests pass/fail, task success/failure)
- **Self-Reflection (M_sr)**: Generates verbal feedback from (task, trajectory, evaluation)
- **Memory (mem)**: Sliding window of past reflections appended to the actor's context

### Results to remember:
- HumanEval (code): 80.1% (the paper's own GPT-4 baseline) -> 91.0% with Reflexion (+10.9pp). The 67% figure sometimes cited is from OpenAI's GPT-4 tech report and shouldn't define the delta.
- ALFWorld (decision-making): 75% -> 97% (+22pp)
- HotPotQA (reasoning): 34% -> 51% (+17pp)

### How to read this paper:
1. **Start with Figure 1** -- it shows the full Reflexion loop visually
2. **Read Section 3** (Method) carefully -- the three components and how they interact
3. **Study Table 1-3** -- the quantitative results across domains
4. **Read Section 5** (Analysis) -- when Reflexion helps and when it doesn't

### Related Papers:
- **Self-Refine** (Madaan et al., 2023): Similar idea but without explicit memory; iterative refinement
- **Constitutional AI** (Bai et al., 2022): Self-critique against principles; training-time version
- **LATS** (Zhou et al., 2023): Combines tree search with Reflexion -- search over trajectories with reflection
- **Self-Debugging** (Chen et al., 2023): Reflexion specifically for code debugging

### Interview question this paper answers:
> "How can you improve an agent's performance at inference time without fine-tuning?"

Answer: Use Reflexion -- let the agent generate a solution, evaluate it against a verifier, generate verbal feedback about what went wrong, store that feedback in memory, and try again. This achieves significant improvements (10-20pp) with just 2-3 reflection rounds, at the cost of 2-3x inference compute.

---

*End of Notebook 16*